In [1]:
from neo4j import GraphDatabase
from igraph import Graph
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import math

In [2]:
g = GraphDatabase.driver('bolt://localhost:11005', auth=('neo4j', '123'))

In [3]:
def get_all_edge():
    with g.session() as session:
        query = ("""
        MATCH (s)-[r]->(t)
        WHERE s.province='Đà Nẵng' AND t.province='Đà Nẵng'
        RETURN s, t, r
        """)
        re = session.run(query).values()
    return re

def get_all_node():
    with g.session() as session:
        query = ("""
        MATCH (n)
        WHERE n.province='Đà Nẵng'
        return n
        """)
        return session.run(query).values()

def date_time(d):
    t = '/'.join([str(d.year), str(d.month), str(d.day)])
    return datetime.strptime(t, '%Y/%m/%d')

edges = get_all_edge()
nodes = get_all_node()

In [4]:
graph = Graph(directed = True)

def gen_nodes(nodes):
    for i in range(len(nodes)):
        v = nodes[i][0]
        lbl = list(v.labels)[0]
        if lbl=='Patient':
            graph.add_vertex(name=v['name'])
            graph.vs[i]['age_group'] = v['age_group']
            graph.vs[i]['full_name'] = v['full_name']
            graph.vs[i]['label'] = lbl
            graph.vs[i]['commune'] = v['commune']
            graph.vs[i]['district'] = v['district']
            graph.vs[i]['province'] = v['province']
            graph.vs[i]['onset_date'] = date_time(v['onset_date'])
            graph.vs[i]['announce_date'] = date_time(v['announce_date'])
            graph.vs[i]['quarantine_date'] = date_time(v['quarantine_date'])
            graph.vs[i]['pagerank'] = 2
        elif lbl=='Location':
            graph.add_vertex(name=v['name'])
            graph.vs[i]['label'] = lbl
            graph.vs[i]['commune'] = v['commune']
            graph.vs[i]['district'] = v['district']
            graph.vs[i]['province'] = v['province']
            graph.vs[i]['l_type'] = v['l_type']

def gen_edge(edges):
    for i in range(len(edges)):
        start_node = edges[i][0]
        end_node = edges[i][1]
        r_type = edges[i][2].type
        graph.add_edge(start_node['name'], end_node['name'], weight = random.random(), r_type = r_type)
        
        
gen_nodes(nodes)
gen_edge(edges)

In [5]:
def get_node_cluster(cluster, node):
    for i in range(len(cluster)):
        if node in cluster[i]:
            return i


def get_color(cluster):
    color = []
    for _ in range(len(cluster)):
        random_number = random.randint(0, 16777215)
        hex_number = str(hex(random_number))
        hex_number = '#' + hex_number[2:]
        color.append(hex_number)
    return color


def get_cluster_list(mode='clusters'):
    if mode=='clusters':
        return graph.clusters()
    elif mode=='age_group':
        return cluster_by_age(graph)
    else:
        return graph.community_optimal_modularity()
    
def cluster_by_age(graph):
    cluster = []
    return cluster


# Layout theo vị trí địa lí

Phần này khó kinh dị.

Viết sơ sơ ý tưởng ở đây để triển khai cho dễ.

## 1. Lấy vị trí địa lý của từng địa phương

Kết quả trả về là (longitude, latitude) của địa điểm.

Cho trước vị trí của Đà Nẵng là 

```python 
danang = [[16.2661, 15.8786], [107.7642, 108.3849]]
```
Vị trí này không quá quan trọng, tuy nhiên nên đủ chính xác vì quá trình scale sau này có thể dẫn đến sai số lớn

```pseudo
for l in location:
    get_latlong(l)
    return (long, lat)
```

Kết quả trả về của $(l, x, y)$ là tọa độ theo longitude và latitude của địa điểm $l$ (xã/phường) thuộc Đà Nẵng.

## Xử lí vị trí của các bệnh nhân

Để xử lí vị trí với các bệnh nhân, cần tạo ra các subgraph mà các nodes là các bệnh nhân ở cùng địa phương. Khi đó cần layout các nodes đó xung quanh địa phương $l$ đã nói ở trên.

### Ý tưởng chung

Địa phương $l$ có tọa độ $(x_l, y_l)$, bản đồ Đà Nẵng được bao trong box có các tọa độ $(long_1, long_2), (lat_1, lat_2)$.

Sẽ cần phải biến các tọa độ (0, 0) và có kích thước (800, 600), thực hiện được qua 2 phép biến hình là tịnh tiến và vị tự. 

Các bệnh nhân sống ở địa điểm $l$ sẽ được layout quanh điểm $(x_l, y_l)$ bằng thuật toán Fruchterman - Reingold, trong đó các node được giới hạn quanh điểm $(x_l+-\epsilon, y_l +- \epsilon)$

### Lấy subgraph

Phân cụm bệnh nhân theo địa phương.

Phân cụm bằng cách lấy tuple (commune, district, province) từ file địa điểm rồi so sánh với từng bệnh nhân, nếu đúng sẽ cùng cụm và được thêm vào list các đỉnh thỏa mãn. Sau đó lấy subgraph rồi thực hiện layout. Sau khi layout sẽ thực hiện gán tọa độ vào x, y

In [9]:
bing_key = 'AovFiLehDbmCtWFshZqJN1hDRcbMPswtqfnG7ytaZL0Wm_c0LKD_1Hf8pUPTq_pw'
bing = Bing(bing_key)

In [10]:
from geopy.geocoders import Here, Bing, Photon

apikey = "uwDDPQK_eOT3uuAaGArNRdj9TRqDwd8c3dMJJDhc-cM"
app_id = "KxVdPXp1uUggmh14yidu"

here = Here(app_id=app_id, apikey=apikey)
photon = Photon()

In [11]:
danang = [[16.2661, 15.8786], [107.7642, 108.3849]]
lx1 = danang[0][0]
lx2 = danang[0][1]

In [12]:
dt = pd.read_csv('location.csv')
dt = pd.DataFrame(dt, columns=['commune','district' ,'province'])
commune = list(dt['commune'])
district = list(dt['district'])
province = list(dt['province'])

commune_list= [[c, d, p] for c, d, p in zip(commune, district, province)] 
district_list = [[d, p] for d, p in zip(district, province)] 
province_list = [[p] for p in province] 

def get_commune_location_list(commune, district, province):
    loc = [tuple([commune[0], district[0], province[0]])]
    for i in range(len(commune)):
        tmp = tuple([commune[i], district[i], province[i]])
        if tmp not in loc:
            loc.append(tmp)
    return loc

def get_district_location_list(district, province):
    loc = [tuple([district[0], province[0]])]
    for i in range(len(district)):
        tmp = tuple([district[i], province[i]])
        if tmp not in loc:
            loc.append(tmp)
    return loc

def get_province_location_list(province):
    loc = [tuple([province[0]])]
    for i in range(len(province)):
        tmp = tuple([province[i]])
        if tmp not in loc:
            loc.append(tmp)
    return loc
    

commune_lst = get_commune_location_list(commune, district, province)
district_lst = get_district_location_list(district, province)
province_lst = get_province_location_list(province)

In [13]:
def get_commune_longlat_by_location(location):
    location = here.geocode("{}, {}, {}".format(location[0], location[1], location[2]))
    return location

def get_district_longlat_by_location(location):
    location = photon.geocode("{}, {}".format(location[0], location[1]))
    return location

def get_province_longlat_by_location(location):
    location = here.geocode("{}".format(location[0]))
    return location

def get_longlat_list(loc, cat):
    pos = {}
    if cat=='commune':
        for location in loc:
            longlat = get_commune_longlat_by_location(location)
            pos[location] = [longlat.longitude, longlat.latitude]
    elif cat=='district':
        for location in loc:
            longlat = get_district_longlat_by_location(location)
            pos[location] = [longlat.longitude, longlat.latitude]
    elif cat=='province':
        for location in loc:
            longlat = get_province_longlat_by_location(location)
            pos[location] = [longlat.longitude, longlat.latitude]
    return pos

l_commune = get_longlat_list(commune_lst, 'commune')
l_district = get_longlat_list(district_lst, 'district')
l_province = get_longlat_list(province_lst, 'province')

In [14]:
location  = photon.geocode('Bình Sơn, Quảng Ngãi')
location.latitude

15.31189165

In [16]:
def get_subgraph_by_location(graph, location):
    vs = []
    for v in graph.vs:
        if v['label']=='Patient':
            l = tuple([v['commune'], v['district'], v['province']])
            if l==location:
                vs.append(v)
    return graph.subgraph(vs)

def layout_subgraph(subgraph, location, epsx = 0.005, epsy = 0.005):
    long = l_commune[location][0]
    lat = l_commune[location][1]
    minx=long-epsx
    maxx=long+epsx
    miny=lat-epsy
    maxy=lat+epsy
    bbox=(minx, miny, maxx, maxy)
    layout=subgraph.layout_random()
    layout.fit_into(bbox)
    return layout

def gen_json(graph, path="../visualization/data.json"):
    gr = {'nodes':[], 'edges':[]}
    cluster = get_cluster_list()
    clr_list = get_color(cluster)
    for location in commune_lst:
        subgraph = get_subgraph_by_location(graph, location)
        pos = layout_subgraph(subgraph, location)
        for v, t in zip(subgraph.vs, pos):
            v_src = graph.vs.find(name=v['name'])
            vertex = {}
            vertex['id'] = v_src.index
            vertex['label'] = v_src['name']
            vertex['x'] = t[0]
#             vertex['y'] = t[1]
            vertex['y'] = lx1 + lx2 - t[1]
            if v['label']=='Patient':
                vertex['full_name'] = v_src['full_name']
                vertex['age_group'] = v_src['age_group']
                vertex['onset_date'] = v_src['onset_date'].strftime('%d/%m/%Y')
                vertex['announce_date'] = v_src['announce_date'].strftime('%d/%m/%Y')
                vertex['quarantine_date'] = v_src['quarantine_date'].strftime('%d/%m/%Y')
                vertex['pagerank'] = v_src.pagerank(weights=graph.es['weight'])
                vertex['size'] = 30*vertex['pagerank']
                vertex['commune'] = v_src['commune']
                vertex['_color'] = clr_list[get_node_cluster(cluster, v_src.index)]
            gr['nodes'].append(vertex)
    for v in graph.vs:
        if v['label']=='Location':
            if v['l_type'] == "district":
                tp = tuple([v['district'], v['province']])
                for location in district_lst:
                    if tp==location:
                        vertex = {}
                        vertex['id']=v.index
                        vertex['type'] = 'square'
                        vertex['pagerank'] = 0.003
                        vertex['size'] = 20*vertex['pagerank']
                        vertex['_color'] = "#444444"
                        vertex['l_type'] = v['l_type']
                        vertex['label'] = v['name']
                        vertex['x'] = l_district[location][0]
#                         vertex['y'] = l_district[location][1]
                        vertex['y'] = lx1 + lx2 - l_district[location][1]
                        gr['nodes'].append(vertex)
                        break
            elif v['l_type'] == "province":
                tp = tuple([v['province']])
                for location in province_lst:
                    if tp==location:
                        vertex = {}
                        vertex['id']=v.index
                        vertex['pagerank'] = 0.004
                        vertex['size'] = 20*vertex['pagerank']
                        vertex['type'] = 'star'
                        vertex['_color'] = "#000000"
                        vertex['l_type'] = v['l_type']
                        vertex['label'] = v['name']
                        vertex['x'] = l_province[location][0]
#                         vertex['y'] = l_province[location][1]
                        vertex['y'] = lx1 + lx2 - l_province[location][1]
                        gr['nodes'].append(vertex)
                        break
            elif v['l_type'] == "commune":
                tp = tuple([v['commune'], v['district'], v['province']])
                for location in commune_lst:
                    if tp==location:
                        vertex = {}
                        vertex['id']=v.index
                        vertex['pagerank'] = 0.004
                        vertex['size'] = 20*vertex['pagerank']
                        vertex['type'] = 'diamond'
                        vertex['_color'] = "#ccc"
                        vertex['l_type'] = v['l_type']
                        vertex['label'] = v['name']
                        vertex['x'] = l_commune[location][0]
#                         vertex['y'] = l_commune[location][1]
                        vertex['y'] = lx1+lx2 - l_commune[location][1]
                        gr['nodes'].append(vertex)
                        break
    for e in graph.es:
        edge = {}
        edge['id'] = e.index
        edge['source'] = e.source
        edge['target'] = e.target
        edge['weight'] = e['weight']
        edge['type'] = e['r_type']
        edge['size'] = 20*e['weight']
        edge['_color'] = '#34c0eb'
        gr['edges'].append(edge)
    try:
        with open(path,'w+', encoding='utf8') as f:
            json.dump(gr, f)
        return True
    except:
        return False
            
gen_json(graph)            

True

In [17]:
def remove_province(graph):
    graph2 = graph
    for v in graph2.vs:
        if v['l_type']=='province':
            graph2.delete_vertices(v)
            continue
    return graph2

graph2 = remove_province(graph)
gen_json(graph2)

True

# Phần này dùng dể sinh trọng số cho các cạnh của đồ thị

Trọng số được đánh tùy vào các cạnh của đồ thị, nói cách khác, trọng số của một cạnh $e$ sẽ phụ thuộc vào:
1. Đỉnh nguồn (source)
2. Đỉnh đích (target)
3. Loại canh (rel type)

Trọng số này thay đổi theo thời gian. Trọng số này thể hiện cho khả năng lây nhiễm từ của node nguồn tới node đích.

Tại thời điểm node nguồn phát bệnh, khả năng lây nhiễm của node này cho những người tiếp xúc là cao nhất, ta sẽ gán bằng 1. Các ngày sau/trước thời điểm đó, trọng số sẽ giảm dần.

- Trước thời điểm phát bệnh, trọng số cần phải thấp bởi lượng virus trong cơ thể chưa đủ để phát bệnh cũng như lây cho người tiếp xúc
- Sau thời điểm phát bệnh, trọng số cũng giảm do lúc này người bệnh và những người xung quanh đã có nhận thức về việc người xung quanh mình có triệu chứng, cũng như sẽ bị bế đi cách li một vài ngày sau đó. Tuy nhiên thường là phát bệnh cái là bị các quan trên bế đi ngay nên có thể coi như node này bị xóa khỏi đồ thị


In [18]:
P =[[1,0.8,0.6,0.4], [0.8,1,0.8,0.6], [0.6,0.8,1,0.8], [0.4,0.6,0.8,1]]
rel = {'UNKNOWN':0.5, 'STAFF_PATIENT': 0.4, 'FELLOW': 0.4, 'RELATIVES': 0.9, 'SOCIAL': 0.4}

In [19]:
def get_edge_from_target_vertex(graph, v):
    edge_list = [edge for edge in graph.es if graph.vs[edge.target]['name']==v['name']]
    return edge_list

def gen_coef(graph, v, e):
    src_group = v['age_group']
    trg_group = graph.vs[e.source]['age_group']
    r_type = e['r_type']
    return P[int(src_group)-1][int(trg_group)-1]*rel[r_type]
    
def gen_weight(graph, v, e, date):
    q_date = v['quarantine_date']
    o_date = v['onset_date']
    coef = gen_coef(graph, v, e)
    d = abs((date - o_date).days)
    e['weight'] = np.e**(coef*d)

In [61]:
date = datetime.strptime('2/8/2020', '%d/%m/%Y')

def get_subgraph_by_date(graph, date):
    graph3 = Graph(directed = True)
    vs = []
    for v in graph.vs:
        if v['label']=='Patient':
            q = (date-v['quarantine_date']).days
            o = (v['onset_date'] - date).days
            if q <= 0 and o < 13: 
                vs.append(v.index)
        elif v['label']=='Location':
            vs.append(v.index)
    graph3 = graph.subgraph(vs)
    return graph3

graph3 = get_subgraph_by_date(graph, date)

gen_json(graph3)

True

In [23]:
def remove_unneccessary_nodes(graph):
    for v in graph.vs:
        if v['label']=='Location':
            if v['l_type'] == 'commune':
                if 'Patient' not in graph.vs[graph.neighborhood(v)]['label']:
                    graph.delete_vertices(v)
            if v['l_type'] == 'district':
                if 'commune' not in graph.vs[graph.neighborhood(v)]['l_type']:
                    graph.delete_vertices(v)
    return graph
                
gen_json(remove_unneccessary_nodes(graph3))

True

In [25]:
def set_weights(graph, date):
    for v in graph.vs:
        if v['label']=='Patient':
            lst = get_edge_from_target_vertex(graph, v)
            for e in lst:
                if graph.vs[e.source]['label']=='Patient':
                    gen_weight(graph, v, e, date)

set_weights(graph3, date)
gen_json(graph3)

True

## Phần này để trả ra rank của các đỉnh

In [ ]:
data = {}
for v in graph.vs:
    if v['label']=='Patient':
        data[v['name']]=[]

def generate_dataframe_by_date(graph_src, graph, date):
    tb = []
    for v in graph_src.vs:
        if v['label']=='Patient':
            try:
                v_tar = graph.vs.find(name=v['name'])
                data[v['name']].append(v_tar.pagerank(weights=graph.es['weight']))
            except:
                data[v['name']].append(0)
                
def generate_dataframe(graph_src, start_date, end_date):
    columns = ['BN']
    delta = timedelta(days=1)
    date = start_date
    while(end_date - date).days>=0:
        graph = get_subgraph_by_date(graph_src, date)
        graph  = remove_unneccessary_nodes(graph)
        set_weights(graph, date)
        generate_dataframe_by_date(graph_src, graph, date)
        date += delta
        columns.append(datetime.strftime(date, '%d/%m/%Y'))
    return columns


start_date = datetime.strptime('1/7/2020', '%d/%m/%Y')
end_date = datetime.strptime('12/8/2020', '%d/%m/%Y')

columns = generate_dataframe(graph, start_date, end_date)

In [73]:
columns

['BN',
 '02/07/2020',
 '03/07/2020',
 '04/07/2020',
 '05/07/2020',
 '06/07/2020',
 '07/07/2020',
 '08/07/2020',
 '09/07/2020',
 '10/07/2020',
 '11/07/2020',
 '12/07/2020',
 '13/07/2020',
 '14/07/2020',
 '15/07/2020',
 '16/07/2020',
 '17/07/2020',
 '18/07/2020',
 '19/07/2020',
 '20/07/2020',
 '21/07/2020',
 '22/07/2020',
 '23/07/2020',
 '24/07/2020',
 '25/07/2020',
 '26/07/2020',
 '27/07/2020',
 '28/07/2020',
 '29/07/2020',
 '30/07/2020',
 '31/07/2020',
 '01/08/2020',
 '02/08/2020',
 '03/08/2020',
 '04/08/2020',
 '05/08/2020',
 '06/08/2020',
 '07/08/2020',
 '08/08/2020',
 '09/08/2020',
 '10/08/2020',
 '11/08/2020',
 '12/08/2020',
 '13/08/2020']